In [ ]:
from main import*
from run_estimator import*
from datasets import cuboid

from IPython.display import clear_output  # To clear tqdm bars

Generate ring point clouds for different number of samples.

In [ ]:
n_samples = np.array([100, 200, 500, 1000, 2000])

avg = 200  # Number of repetitions for averaging stats
avg_full = 10  # Smaller for the full graph

seed = 16  # Random seed

points = []  # Datasets
X_init = [[-0.75, 0], [0.75, 0]]  # Endpoints to be forced into the dataset
ranges = np.array([1, 0.25])
for n in n_samples:
    points.append([cuboid(n, ranges, X_init=X_init, seed=seed+a) for a in range(avg)])

In [ ]:
color_points = 'gray'
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(points[-1][-1][:,0], points[-1][-1][:,1], c=color_points, s=20, marker='x')
ax.axis('off')
ax.set_aspect('equal', adjustable='box')

ax.set_title(f'Example of dataset, n={n_samples[-1]}')
plt.gcf().set_dpi(100)
plt.show()

Experiments

In [ ]:
experiment = {}  # To store all the data

DTM parameters

In [ ]:
m = 0.05
p = 2
beta = 2
dtm_arg = DTM_arg(m, p, beta)
dtms = [[DTM(X, dtm_arg) for X in Xs] for Xs in points]

precision = 5  # Number of points to approximate the integral over edges
precisions = np.full(len(points), precision)

Full graph

In [ ]:
name = 'full graph'

experiment[name] = run_estimator(FDTM, points, dtms=dtms, precisions=precisions, avg=avg_full)

experiment[name].color = 'red'
experiment[name].linestyle = '-'

clear_output()

kNN with $k=n^{1/2}$

In [ ]:
knns = np.array(n_samples**0.5, dtype=int)  # Number of nearest neighbours

name = f'k = sqrt(n)'

experiment[name] = run_estimator(FDTM, points, knns=knns, dtms=dtms, precisions=precisions, avg=avg)

experiment[name].color = 'orange'
experiment[name].linestyle = '-'

clear_output()

kNN with $k=\log(n)$

In [ ]:
knns = np.array(np.maximum(1, np.log(n_samples)), dtype=int)  # Number of nearest neighbours

name = 'k = log(n)'

experiment[name] = run_estimator(FDTM, points, knns=knns, dtms=dtms, precisions=precisions, avg=avg)

experiment[name].color = 'green'
experiment[name].linestyle = '-'

clear_output()

True FDTM length

In [ ]:
length = norm(np.array(X_init[0]) - np.array(X_init[1]))  # Euclidean length
omega = np.pi  # unit ball volume
dtm_constant = (np.prod(2*ranges) * m / omega)**(beta/2) * (2 / (p+2))**(beta/p)
true_length = length * dtm_constant  # Euclidean length times the constant DTM. Warning : this no longer holds if m is too large.
print(f"True length : {true_length}")

Plots

In [ ]:
save = True  # Save figures

In [ ]:
fig, ax = plt.subplots()

lines = []  # Save plots to reorder them in legend
line, = ax.plot(n_samples, np.full(n_samples.shape, true_length), label='true distance', c='black', linestyle='dotted', linewidth=4)  # True distance as a reference
lines.append(line)

for name, data in experiment.items():
    distances_nan = np.where(np.isinf(data.distances), np.nan, data.distances)  # Convert `inf` to NaN to disregard the values in plots
    mean_distances = np.nanmean(distances_nan, axis=-1)  # Average distances over repetition
    plot_kwargs = {'label' : name, 'c' : data.color, 'linestyle' : data.linestyle, 'marker' : 'o'}
    
    line, = ax.plot(n_samples, mean_distances, **plot_kwargs)
    lines.append(line)

ax.set_xlabel('Number of sample points')
ax.set_xscale('log')
ax.set_ylabel('Distance')
ax.set_ylim(0.01, 0.018)
ax.grid(True)
lines = lines[1:] + [lines[0]]  # To reorder labels in legend
labels = [h.get_label() for h in lines]
ax.legend(lines, labels)

if save:
    fig.savefig(f"figures\\knn_estimated_distance.png", dpi=300)

ax.set_title('Estimated distance')
plt.show()

In [ ]:
fig, ax = plt.subplots()

for name, data in experiment.items():
    distances_nan = np.where(np.isinf(data.distances), np.nan, data.distances)  # Convert `inf` to NaN to disregard the values in plots
    durations = np.array(data.durations)
    plot_kwargs = {'label' : name, 'c' : data.color, 'linestyle' : data.linestyle, 'marker' : 'o'}

    error = np.abs(distances_nan - true_length)
    mean, std = np.nanmean(error, axis=1), np.nanstd(error, axis=1)

    ax.loglog(durations, mean, **plot_kwargs)

ax.set_xlabel('Duration (s)')
ax.set_ylabel('Mean error')
ax.set_ylim(10**(-4), 10**(-2))
ax.grid(True)
ax.legend()

if save:
    fig.savefig(f"figures\\knn_error_vs_runtime.png", dpi=300)

ax.set_title('Estimation error per runtime duration')
plt.show()